# 성균관대 자연과학캠퍼스 1398피트니스센터 · 운동기구 인식 첫 모델

**실행 목표:** 공개 운동기구 데이터를 이용해 **5종 객체탐지 기준 모델**을 만들어 학습·평가하고, 현장 사진으로 별도 검증할 준비를 합니다.

> **반드시 구분:** 이것은 일반 헬스장 사진으로 학습한 **공개 데이터 기반 proxy 모델**입니다. 1398피트니스센터의 실제 기구와 외형이 일치한다고 검증되지 않았습니다. 센터 촬영 사진을 추가하고, 독립 테스트를 해야 현장 인식 성능을 말할 수 있습니다.

첫 5종: `treadmill / seated_chest_press / shoulder_press / seated_leg_press / smith_machine`

데이터 출처: [FitFuel / All Gym Equipment v2](https://universe.roboflow.com/fitfuel/all-gym-equipment/dataset/2), Roboflow Universe, **CC BY 4.0**. 이 데이터의 원래 범주인 `Chest Press Machine`, `Leg Press Machine`은 센터의 **SEATED** 세부 형태와 정확히 대응하지 않을 수 있습니다.

데이터는 Roboflow 계정의 **Private API Key**로 다운로드하거나, 본인이 준비한 YOLO ZIP 파일로 시작합니다. 키는 **Colab 비밀 키 메뉴**의 `ROBOFLOW_API_KEY`에 넣으며, 코드 셀에 직접 적지 않습니다. 사진과 라벨은 서버/Drive에 저장하지만 사용자 사진을 Roboflow에 자동 업로드하지 않습니다.

실행 순서: ① 환경/Drive → ② 데이터 다운로드 → ③ 5종 정리·정답 검수 → ④ 학습 → ⑤ 평가·추론 → ⑥ 모바일 변환(옵션) → ⑦ Drive 결과 보관.

## 1. 설정: 첫 실행은 반드시 `smoke`부터

- `smoke`: 소량의 공개 데이터를 학습하여 연결만 검증. 성능 수치를 보고하지 않습니다.
- `baseline`: 데이터 검수 후 전체 학습·평가. 이것도 **센터 실환경 성능이 아닙니다**.
- `DATASET_ZIP_NAME`: Roboflow 대신 이미 갖고 있는 YOLO 데이터 ZIP을 Drive의 `02_dataset` 폴더에 넣었다면 파일명으로 설정합니다.
- `RUN_MOBILE_EXPORT`: 실제 학습 완료 후 `True`로 바꿔 LiteRT 변환을 진행합니다.

In [ ]:
RUN_MODE = "smoke"             # "smoke" 또는 "baseline"
DATASET_ZIP_NAME = None       # 예: "my_center_yolo_dataset.zip" (Drive/02_dataset)
RUN_MOBILE_EXPORT = False     # 모델 학습 후 True로 바꿀 수 있음
RUN_PHOTO_UPLOAD = False      # 본인 사진을 직접 넣어 예측하려면 True
DISPLAY_CONFIDENCE = 0.25
IMAGE_SIZE = 320 if RUN_MODE == "smoke" else 640
EPOCHS = 2 if RUN_MODE == "smoke" else 30
BATCH = 8 if RUN_MODE == "smoke" else 16
SEED = 42
GOOGLE_DRIVE_ROOT_FOLDER = "SKKU_1398_Fitness_ML"
assert RUN_MODE in ("smoke", "baseline")
print("설정:", RUN_MODE, "입력 크기:", IMAGE_SIZE, "epochs:", EPOCHS)

## 2. 패키지 설치 및 GPU 확인

유료 Colab이어도 GPU가 자동으로 붙지는 않습니다. **런타임 → 런타임 유형 변경**에서 GPU를 선택한 후 실행하세요. 첫 실험은 CPU에서도 동작할 수 있으나 느릴 수 있습니다.

In [ ]:
import os, sys, subprocess
subprocess.check_call([sys.executable,"-m","pip","install","-q", "ultralytics>=8.3,<9", "roboflow", "PyYAML", "Pillow"])
import torch
from ultralytics import YOLO
print("Python",sys.version.split()[0],"| torch",torch.__version__,"| CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
else:
    print("주의: GPU가 확인되지 않았습니다. smoke 모드만 권장합니다.")

## 3. Google Drive 연결

여기서 나타나는 Google 인증창에서는 **본인이 계정과 접근 권한을 확인**해야 합니다. 인증 코드나 비밀번호를 채팅에 입력하지 마세요.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")/GOOGLE_DRIVE_ROOT_FOLDER
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f"프로젝트 Drive 폴더가 보이지 않습니다: {DRIVE_ROOT}. 구글 계정 및 Drive 마운트를 확인하세요.")
for folder in ("01_notebooks", "02_dataset", "03_training_runs", "04_exports_and_reports"):
    (DRIVE_ROOT/folder).mkdir(parents=True,exist_ok=True)
WORK=Path("/content/skku1398_workspace")
WORK.mkdir(parents=True,exist_ok=True)
RUN_NAME=f"skku1398_public_proxy_{RUN_MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DRIVE_RUN=DRIVE_ROOT/"03_training_runs"/RUN_NAME
DRIVE_RUN.mkdir(parents=True,exist_ok=True)
print("Drive 작업 위치:", DRIVE_ROOT)
print("이번 실험 보관 위치:", DRIVE_RUN)

## 4. 내장된 데이터 가공 코드 설치

별도 Python 파일 없이 **이 노트북 하나만** 올려도 실행할 수 있도록 코드를 내장했습니다. ZIP 패키지의 `src/skku1398_ml.py`와 동일한 모듈입니다.

In [ ]:
LIB_CODE = '"""SKKU 1398 Fitness Center: YOLO dataset preparation and audit utilities.\n\nThis code prepares a public *proxy* dataset for five provisional center classes.\nIt never asserts the public class names perfectly match the installed machines.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport json\nimport random\nimport re\nimport shutil\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport yaml\nfrom PIL import Image, ImageDraw, ImageOps\n\nCLASS_NAMES = [\n    "treadmill",\n    "seated_chest_press",\n    "shoulder_press",\n    "seated_leg_press",\n    "smith_machine",\n]\n\n# The public names can be broader than the equipment listed by the center.\nSOURCE_ALIASES = {\n    "treadmill": ["Treadmill", "Tredmill"],\n    "seated_chest_press": ["Chest Press Machine", "Chest Press", "Seated Chest Press"],\n    "shoulder_press": ["Shoulder Press Machine", "Shoulder Press"],\n    "seated_leg_press": ["Leg Press Machine", "Leg Press", "Seated Leg Press"],\n    "smith_machine": ["Smith Machine", "Smith"],\n}\nMATCH_NOTE = {\n    "treadmill": "generic treadmill vs center treadmill: appearance unverified",\n    "seated_chest_press": "generic chest press may include other press configurations",\n    "shoulder_press": "generic shoulder press vs exact installed machine unverified",\n    "seated_leg_press": "generic leg press may include horizontal/plate-loaded variants",\n    "smith_machine": "generic smith machine vs exact installed unit unverified",\n}\n\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}\nSPLIT_ALIASES = {"train": ("train",), "val": ("val", "valid", "validation"), "test": ("test",)}\n\n\ndef normalize_name(text: str) -> str:\n    return re.sub(r"[^a-z0-9]+", "", str(text).lower())\n\n\ndef parse_class_names(raw: object) -> list[str]:\n    if isinstance(raw, list):\n        return [str(v) for v in raw]\n    if isinstance(raw, dict):\n        indexed = {int(k): str(v) for k, v in raw.items()}\n        if sorted(indexed) != list(range(len(indexed))):\n            raise ValueError("Class IDs in YAML must start at 0 and be contiguous")\n        return [indexed[i] for i in range(len(indexed))]\n    raise ValueError("data.yaml must have a list/dictionary called names")\n\n\ndef source_to_target(source_names: list[str]) -> dict[int, int]:\n    alias_lookup = {}\n    for target_id, canonical in enumerate(CLASS_NAMES):\n        for alias in [canonical] + SOURCE_ALIASES[canonical]:\n            norm = normalize_name(alias)\n            if norm in alias_lookup and alias_lookup[norm] != target_id:\n                raise ValueError(f"Ambiguous class alias {alias}")\n            alias_lookup[norm] = target_id\n    found = {}\n    for src_id, name in enumerate(source_names):\n        norm = normalize_name(name)\n        if norm in alias_lookup:\n            found[src_id] = alias_lookup[norm]\n    return found\n\n\ndef read_yolo_boxes(label_file: Path, map_ids: dict[int, int]) -> tuple[list[tuple[int, float, float, float, float]], list[str]]:\n    """Return selected valid boxes and issue messages. Unmapped classes are ignored."""\n    boxes, errors = [], []\n    if not label_file.exists():\n        return boxes, ["missing label file"]\n    for line_num, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), 1):\n        line = line.strip()\n        if not line:\n            continue\n        fields = line.split()\n        if len(fields) != 5:\n            errors.append(f"line {line_num}: expected 5 values")\n            continue\n        try:\n            fcls = float(fields[0])\n            if not fcls.is_integer():\n                raise ValueError("non-integer class ID")\n            src_cls = int(fcls)\n            vals = [float(v) for v in fields[1:]]\n        except ValueError:\n            errors.append(f"line {line_num}: failed to parse numeric box")\n            continue\n        if src_cls not in map_ids:\n            continue\n        cx, cy, bw, bh = vals\n        if not all(map(lambda x: float(\'-inf\') < x < float(\'inf\'), vals)):\n            errors.append(f"line {line_num}: NaN/Inf box")\n            continue\n        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):\n            errors.append(f"line {line_num}: values outside YOLO normalized range")\n            continue\n        if not (0 <= cx-bw/2 and cx+bw/2 <= 1 and 0 <= cy-bh/2 and cy+bh/2 <= 1):\n            errors.append(f"line {line_num}: box crosses image boundary")\n            continue\n        boxes.append((map_ids[src_cls], cx, cy, bw, bh))\n    return boxes, errors\n\n\ndef _candidate_image_dirs(root: Path, config: dict, split: str) -> list[Path]:\n    cands = []\n    cfgvalue = config.get(split)\n    if split == \'val\':\n        cfgvalue = cfgvalue or config.get(\'valid\') or config.get(\'validation\')\n    if cfgvalue:\n        if isinstance(cfgvalue, list):\n            if len(cfgvalue) != 1:\n                raise ValueError(f"split {split}: multiple datasets not supported in baseline")\n            cfgvalue = cfgvalue[0]\n        candidate = Path(str(cfgvalue))\n        if candidate.is_absolute():\n            cands.append(candidate)\n        else:\n            cands.append(root / candidate)\n            cands.append(root / Path(str(cfgvalue).replace("../", "")))\n    for synonym in SPLIT_ALIASES[split]:\n        cands += [root / synonym / "images", root / synonym]\n    return cands\n\n\ndef detect_image_dir(root: Path, config: dict, split: str) -> Path | None:\n    for candidate in _candidate_image_dirs(root, config, split):\n        if candidate.is_dir() and any(p.suffix.lower() in IMAGE_SUFFIXES for p in candidate.iterdir() if p.is_file()):\n            return candidate.resolve()\n    return None\n\n\ndef find_yaml(root: Path) -> Path:\n    direct = root / "data.yaml"\n    if direct.exists():\n        return direct\n    found = [p for p in root.rglob("data.yaml") if len(p.relative_to(root).parts) <= 3]\n    if len(found) != 1:\n        raise FileNotFoundError(f"Expected one data.yaml within {root}; found {len(found)}")\n    return found[0]\n\n\ndef _label_path(image_path: Path) -> Path:\n    if image_path.parent.name.lower() == "images":\n        return image_path.parent.parent / "labels" / (image_path.stem + ".txt")\n    return image_path.parent / "labels" / (image_path.stem + ".txt")\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as fh:\n        for chunk in iter(lambda: fh.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef dhash(path: Path) -> int:\n    with Image.open(path) as image:\n        image = ImageOps.exif_transpose(image).convert(\'L\').resize((9,8))\n        arr = list(image.get_flattened_data()) if hasattr(image, \'get_flattened_data\') else list(image.getdata())\n    result = 0\n    for y in range(8):\n        for x in range(8):\n            result = (result << 1) | int(arr[y*9+x] > arr[y*9+x+1])\n    return result\n\n\ndef _stratified_pick(rows: list[dict], limit: int | None, seed: int = 42) -> list[dict]:\n    """Select examples with every available class represented before filling randomly."""\n    if limit is None or len(rows) <= limit:\n        return sorted(rows, key=lambda r: r[\'source_image\'])\n    rnd = random.Random(seed)\n    order = rows.copy()\n    rnd.shuffle(order)\n    picked, seen = [], set()\n    for class_id in range(len(CLASS_NAMES)):\n        possible = [r for r in order if any(b[0] == class_id for b in r[\'boxes\']) and r[\'source_image\'] not in seen]\n        if possible and len(picked) < limit:\n            row = possible[0]\n            picked.append(row)\n            seen.add(row[\'source_image\'])\n    for row in order:\n        if len(picked) >= limit:\n            break\n        if row[\'source_image\'] not in seen:\n            picked.append(row)\n            seen.add(row[\'source_image\'])\n    return picked\n\n\ndef _write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as f:\n        writer = csv.DictWriter(f, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows([{k: r.get(k, \'\') for k in fields} for r in rows])\n\n\ndef prepare_yolo_dataset(source_root: Path | str, dest_root: Path | str, *,\n                         mode: str = "smoke", max_train: int = 200,\n                         max_val: int = 60, max_test: int = 40,\n                         seed: int = 42) -> dict:\n    """Prepare five-class YOLO data retaining source splits, auditing exact leakage.\n\n    Training and validation may be augmented source images. A separate held-out\n    center-photo evaluation is still required; this routine never claims otherwise.\n    """\n    source_root, dest_root = Path(source_root).resolve(), Path(dest_root).resolve()\n    if mode not in {"smoke", "baseline"}:\n        raise ValueError("mode must be \'smoke\' or \'baseline\'")\n    config_path = find_yaml(source_root)\n    src_root = config_path.parent\n    config = yaml.safe_load(config_path.read_text(encoding=\'utf-8\'))\n    source_names = parse_class_names(config[\'names\'])\n    mapping = source_to_target(source_names)\n    absent = [name for idx, name in enumerate(CLASS_NAMES) if idx not in set(mapping.values())]\n    if absent:\n        raise ValueError(f"Source dataset has no matching classes: {absent}; source names: {source_names}")\n    if dest_root.exists():\n        raise FileExistsError(f"Destination already exists: {dest_root}; choose a fresh directory")\n    dest_root.mkdir(parents=True)\n    (dest_root / \'reports\').mkdir()\n    manifest, issues, near = [], [], []\n    counts = {}\n    # Process more trustworthy held-out files before train to remove exact split leakage.\n    seen_sha: dict[str, dict] = {}\n    seen_hashes: list[tuple[int, str, str]] = []\n    for split in (\'test\', \'val\', \'train\'):\n        image_dir = detect_image_dir(src_root, config, split)\n        if image_dir is None:\n            if split == \'test\':\n                raise FileNotFoundError(\'No test/images found in source. Reserve an independent test split first.\')\n            raise FileNotFoundError(f\'Missing {split}/images in {src_root}\')\n        all_rows, negatives = [], []\n        for img in sorted(image_dir.iterdir()):\n            if not img.is_file() or img.suffix.lower() not in IMAGE_SUFFIXES:\n                continue\n            labels, errors = read_yolo_boxes(_label_path(img), mapping)\n            if errors:\n                issues.append({\'split\':split,\'image\':str(img),\'issue\':\'; \'.join(errors)})\n            if labels:\n                all_rows.append({\'source_image\':str(img),\'boxes\':labels})\n            elif not errors and _label_path(img).exists():\n                negatives.append({\'source_image\':str(img),\'boxes\':[]})\n        if not all_rows:\n            raise ValueError(f\'No selected target boxes in split {split}\')\n        limit = {\'train\':max_train,\'val\':max_val,\'test\':max_test}[split] if mode==\'smoke\' else None\n        pos_limit = max(5,int(limit*0.85)) if limit is not None else None\n        chosen = _stratified_pick(all_rows, pos_limit, seed=seed + len(split))\n        # Include a modest number of negative scenes containing only non-target classes.\n        neg_limit = min(len(negatives), max(1, (limit or len(chosen))//8))\n        random.Random(seed+42).shuffle(negatives)\n        chosen += negatives[:neg_limit]\n        rows_kept=0\n        for row in chosen:\n            img=Path(row[\'source_image\'])\n            sha=_sha256(img)\n            if sha in seen_sha:\n                issues.append({\'split\':split,\'image\':str(img),\'issue\':f\'exact duplicate dropped; first in {seen_sha[sha]["split"]}\'})\n                continue\n            # Validate the image, not only filename and annotation text.\n            try:\n                with Image.open(img) as im:\n                    im.verify()\n            except Exception as ex:\n                issues.append({\'split\':split,\'image\':str(img),\'issue\':f\'image decode failed: {type(ex).__name__}\'})\n                continue\n            prefix = hashlib.sha256(str(img.relative_to(src_root)).encode()).hexdigest()[:14]\n            new_stem = prefix+\'_\'+img.stem[:50]\n            out_image=dest_root/split/\'images\'/f\'{new_stem}{img.suffix.lower()}\'\n            out_label=dest_root/split/\'labels\'/f\'{new_stem}.txt\'\n            out_image.parent.mkdir(parents=True,exist_ok=True)\n            out_label.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(img,out_image)\n            out_label.write_text((\'\\n\'.join(\' \'.join([str(b[0])]+[f\'{n:.6f}\' for n in b[1:]]) for b in row[\'boxes\'])+\'\\n\') if row[\'boxes\'] else \'\',encoding=\'utf-8\')\n            seen_sha[sha]={\'split\':split,\'image\':str(img)}\n            h=dhash(img)\n            seen_hashes.append((h,split,str(img)))\n            rows_kept+=1\n            manifest.append({\'split\':split,\'image\':str(out_image.relative_to(dest_root)),\n                             \'source_image\':str(img),\'sha256\':sha,\'dhash\':f\'{h:016x}\',\n                             \'classes\':\',\'.join(CLASS_NAMES[c] for c in sorted(set(b[0] for b in row[\'boxes\']))),\n                             \'box_count\':len(row[\'boxes\'])})\n        counts[split]={\'available_positive_images\':len(all_rows),\'available_negative_images\':len(negatives),\'selected_before_exact_dedup\':len(chosen),\'kept_images\':rows_kept}\n        if rows_kept<1:\n            raise ValueError(f\'No valid images kept for {split}\')\n    # Report suspicious near-identical images across splits (do NOT auto-delete).\n    if len(seen_hashes)<=6000:\n        for i,(a,sa,ia) in enumerate(seen_hashes):\n            for b,sb,ib in seen_hashes[i+1:]:\n                if sa==sb:\n                    continue\n                d=(a^b).bit_count()\n                if d<=3:\n                    near.append({\'split_a\':sa,\'image_a\':ia,\'split_b\':sb,\'image_b\':ib,\'dhash_distance\':d})\n                    if len(near)>=250:\n                        break\n            if len(near)>=250:break\n    _write_csv(dest_root/\'reports\'/\'manifest.csv\',manifest,[\'split\',\'image\',\'source_image\',\'sha256\',\'dhash\',\'classes\',\'box_count\'])\n    _write_csv(dest_root/\'reports\'/\'label_issues.csv\',issues,[\'split\',\'image\',\'issue\'])\n    _write_csv(dest_root/\'reports\'/\'potential_near_duplicates.csv\',near,[\'split_a\',\'image_a\',\'split_b\',\'image_b\',\'dhash_distance\'])\n    target_yaml={\'path\':str(dest_root),\'train\':\'train/images\',\'val\':\'val/images\',\'test\':\'test/images\',\n                 \'names\':{k:v for k,v in enumerate(CLASS_NAMES)}}\n    (dest_root/\'data.yaml\').write_text(yaml.safe_dump(target_yaml,allow_unicode=True,sort_keys=False),encoding=\'utf-8\')\n    class_boxes=defaultdict(Counter)\n    for r in manifest:\n        labels=(dest_root/Path(r[\'image\']).parent.parent/\'labels\'/(Path(r[\'image\']).stem+\'.txt\'))\n        for line in labels.read_text(encoding=\'utf-8\').splitlines():\n            if line.strip():\n                class_boxes[r[\'split\']][CLASS_NAMES[int(line.split()[0])]]+=1\n    summary={\n        \'classes\':CLASS_NAMES,\'source_root\':str(src_root), \'source_classes\':source_names,\n        \'id_mapping\':{str(i):{\'source_name\':source_names[i], \'target\':CLASS_NAMES[j]} for i,j in mapping.items()},\n        \'match_caveats\':MATCH_NOTE, \'source_split_counts\':counts,\n        \'box_counts_by_split\':{s:dict(class_boxes[s]) for s in (\'train\',\'val\',\'test\')},\n        \'label_issue_count\':len(issues),\'near_duplicate_warning_count\':len(near),\n        \'limitations\':[\'Public 29-class labels are proxy labels, not installed-equipment confirmation\',\n                       \'Near duplicates must be manually reviewed; hash check is not proof of independence\',\n                       \'Evaluate separately on real SKKU 1398 photos from independent capture sessions\']}\n    (dest_root/\'reports\'/\'dataset_audit.json\').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    return summary\n\n\ndef make_annotation_preview(prepared_root: str|Path, output: str|Path, *,max_images:int=9, seed:int=42) -> Path:\n    root=Path(prepared_root)\n    images=list((root/\'train\'/\'images\').glob(\'*\'))\n    images=[p for p in images if p.suffix.lower() in IMAGE_SUFFIXES]\n    if not images:\n        raise FileNotFoundError(\'No prepared train images\')\n    images.sort()\n    random.Random(seed).shuffle(images)\n    images=images[:max_images]\n    tiles=[]\n    for img in images:\n        with Image.open(img) as raw:\n            im=ImageOps.exif_transpose(raw).convert(\'RGB\')\n        lab=root/\'train\'/\'labels\'/f\'{img.stem}.txt\'\n        draw=ImageDraw.Draw(im)\n        for line in lab.read_text().splitlines():\n            c,x,y,w,h=map(float,line.split())\n            W,H=im.size\n            xy=[(x-w/2)*W,(y-h/2)*H,(x+w/2)*W,(y+h/2)*H]\n            draw.rectangle(xy,outline=(255,168,17),width=max(2,W//180))\n            draw.text((xy[0]+4,max(0,xy[1]-18)),CLASS_NAMES[int(c)],fill=(255,168,17))\n        im.thumbnail((430,315))\n        tile=Image.new(\'RGB\',(450,345),\'white\')\n        tile.paste(im,((450-im.width)//2,10))\n        tiles.append(tile)\n    cols=min(3,len(tiles))\n    rows=(len(tiles)+cols-1)//cols\n    collage=Image.new(\'RGB\',(cols*450,rows*345),(244,244,244))\n    for i,tile in enumerate(tiles):\n        collage.paste(tile,((i%cols)*450,(i//cols)*345))\n    out=Path(output)\n    out.parent.mkdir(parents=True,exist_ok=True)\n    collage.save(out)\n    return out\n\n\ndef predictions_to_json(result, *, threshold: float=0.25) -> dict:\n    boxes=[]\n    for box in result.boxes:\n        score=float(box.conf.item())\n        if score<threshold:\n            continue\n        cls=int(box.cls.item())\n        x1,y1,x2,y2=[round(float(t),2) for t in box.xyxy[0].tolist()]\n        boxes.append({\'class_id\':cls,\'class_name\':str(result.names[cls]),\'confidence\':round(score,4),\n                      \'bbox_xyxy_pixels\':[x1,y1,x2,y2]})\n    return {\'status\':\'ok\' if boxes else \'no_detections\',\n            \'image_height\':int(result.orig_shape[0]),\'image_width\':int(result.orig_shape[1]),\n            \'detections\':boxes}\n'
from pathlib import Path
LIB_PATH=Path('/content/skku1398_ml.py')
LIB_PATH.write_text(LIB_CODE,encoding='utf-8')
import sys
sys.path.insert(0,'/content')
from skku1398_ml import CLASS_NAMES, prepare_yolo_dataset, make_annotation_preview, predictions_to_json
print("모델 5개 클래스:",CLASS_NAMES)

## 5. 공개 데이터 또는 Drive ZIP 다운로드

Roboflow Universe는 데이터를 제공하지만 **계정별 다운로드 인증이 필요할 수 있습니다**. Colab 열쇠(비밀 키) 메뉴에 `ROBOFLOW_API_KEY`가 없으면 다운로드를 중단하고 명확한 오류를 보여줍니다. 또는 데이터셋을 YOLO 포맷 ZIP으로 내려받아 Drive `02_dataset`에 넣고 `DATASET_ZIP_NAME`을 지정하면 됩니다.

In [ ]:
from pathlib import Path
import shutil, zipfile
RAW_ROOT=WORK/'raw_source'
RAW_ROOT.mkdir(parents=True,exist_ok=True)

def safe_unzip(zip_path: Path, dest: Path) -> None:
    dest=dest.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            target=(dest/info.filename).resolve()
            if target!=dest and dest not in target.parents:
                raise ValueError('ZIP contains an unsafe path')
        zf.extractall(dest)

if DATASET_ZIP_NAME:
    zip_input=DRIVE_ROOT/'02_dataset'/DATASET_ZIP_NAME
    if not zip_input.is_file():
        raise FileNotFoundError(f"Drive에 다음 ZIP이 보이지 않습니다: {zip_input}")
    local_zip=WORK/zip_input.name
    shutil.copy2(zip_input,local_zip)
    safe_unzip(local_zip,RAW_ROOT)
    SOURCE_ROOT=RAW_ROOT
    print('사용자 YOLO ZIP 불러오기 완료:',local_zip)
else:
    from google.colab import userdata
    try:
        ROBOFLOW_KEY=userdata.get('ROBOFLOW_API_KEY')
    except Exception as exc:
        ROBOFLOW_KEY=None
    if not ROBOFLOW_KEY:
        raise RuntimeError('Colab 비밀 키 메뉴에 ROBOFLOW_API_KEY를 등록하거나 DATASET_ZIP_NAME을 지정하세요.')
    from roboflow import Roboflow
    rf=Roboflow(api_key=ROBOFLOW_KEY)
    dataset=rf.workspace('fitfuel').project('all-gym-equipment').version(2).download('yolov8',location=str(RAW_ROOT/'fitfuel_v2'))
    SOURCE_ROOT=Path(dataset.location)
    del ROBOFLOW_KEY  # 로그에 키를 출력하지 않음
    print('공개 데이터 다운로드 완료:',SOURCE_ROOT)

## 6. 클래스 정리, 중복 검사, 정답 데이터 생성

- 원본 라벨에서 지정한 5종만 추출합니다.
- 드물게 있는 정확히 동일한 파일은 **학습/검증/테스트에 중복 사용하지 않게 제거**합니다.
- 유사 이미지는 별도 CSV로 남기므로 반드시 육안 검토하세요. 자동 중복 검사만으로 독립성이 보장되는 것은 아닙니다.
- 라벨에 문제가 있으면 `label_issues.csv`에 기록합니다.
- 이 공개 데이터는 `Chest Press Machine` 등의 일반 클래스이므로 **센터 기구와 일치하지 않을 수 있습니다**.

In [ ]:
import json
PREPARED=WORK/'prepared'/RUN_NAME
report=prepare_yolo_dataset(
    SOURCE_ROOT, PREPARED,
    mode=RUN_MODE, max_train=200, max_val=60, max_test=40, seed=SEED
)
print(json.dumps(report['source_split_counts'],ensure_ascii=False,indent=2))
print('클래스별 박스:',json.dumps(report['box_counts_by_split'],ensure_ascii=False,indent=2))
print('오류 행 수:',report['label_issue_count'])
print('유사 중복 경고:',report['near_duplicate_warning_count'])
print('원본 → 5종 매핑:',json.dumps(report['id_mapping'],ensure_ascii=False,indent=2))
for f in (PREPARED/'reports').iterdir():
    if f.is_file(): shutil.copy2(f, DRIVE_RUN/f.name)
shutil.copy2(PREPARED/'data.yaml',DRIVE_RUN/'data_reference.yaml')

## 7. 실제 정답 박스 검수

아래 이미지에서 **물체와 정답 상자가 일치하는지** 확인하세요. 심각한 오라벨, 너무 작은 박스, 누락이 많으면 학습 전에 데이터를 수정해야 합니다. 이 단계의 출력은 모델 예측이 아니라 **원본 라벨 검수 그림**입니다.

In [ ]:
from IPython.display import display
from PIL import Image
PREVIEW=make_annotation_preview(PREPARED,WORK/'annotation_review.jpg',max_images=9)
display(Image.open(PREVIEW))
shutil.copy2(PREVIEW,DRIVE_RUN/'annotation_review.jpg')

## 8. YOLO11n 학습

`smoke`는 2회만 학습하며 **품질 지표로 사용하지 마세요**. `baseline`은 전체 공개 데이터를 이용하므로 시간이 더 걸립니다. 학습 중간 가중치는 5 epochs마다 Drive에 백업합니다.

In [ ]:
import shutil, torch, json
from pathlib import Path
from ultralytics import YOLO
MODEL=YOLO('yolo11n.pt')
DEVICE=0 if torch.cuda.is_available() else 'cpu'
if DEVICE=='cpu' and RUN_MODE=='baseline':
    print('경고: baseline을 CPU에서 학습하면 매우 느릴 수 있습니다.')

def backup_to_drive(trainer):
    if (trainer.epoch+1)%5==0 or (trainer.epoch+1)==EPOCHS:
        path=Path(trainer.last)
        if path.exists():
            shutil.copy2(path, DRIVE_RUN/'last_checkpoint.pt')
MODEL.add_callback('on_model_save',backup_to_drive)
training=MODEL.train(
    data=str(PREPARED/'data.yaml'),
    imgsz=IMAGE_SIZE, epochs=EPOCHS,
    batch=BATCH if DEVICE != 'cpu' else 4,
    seed=SEED, device=DEVICE,
    workers=2, cache=False, patience=12 if RUN_MODE=='baseline' else 2,
    project=str(WORK/'runs'),name=RUN_NAME,exist_ok=False,
    plots=True, verbose=True,
)
TRAIN_DIR=Path(MODEL.trainer.save_dir)
BEST=TRAIN_DIR/'weights'/'best.pt'
if not BEST.exists():
    raise FileNotFoundError('학습 종료 후 best.pt 파일이 생성되지 않았습니다')
shutil.copy2(BEST,DRIVE_RUN/'best.pt')
for filename in ('results.csv','results.png','args.yaml','confusion_matrix.png','confusion_matrix_normalized.png'):
    f=TRAIN_DIR/filename
    if f.exists(): shutil.copy2(f,DRIVE_RUN/f.name)
print('학습된 모델 저장:',DRIVE_RUN/'best.pt')

## 9. 별도 분할 평가

여기서 나오는 수치는 **FitFuel 공개 데이터 내 평가 결과**입니다. 1398피트니스센터 현장 성능이 아닙니다. `smoke` 결과는 연결 시험용입니다. 테스트는 모델 선정 이후 한 번만 수행하고, 이 값을 보고 가중치나 임계값을 재선정하지 마세요.

In [ ]:
from ultralytics import YOLO
import json
MODEL_FINAL=YOLO(str(DRIVE_RUN/'best.pt'))
val_metrics=MODEL_FINAL.val(data=str(PREPARED/'data.yaml'),split='test',
                            imgsz=IMAGE_SIZE,batch=8,device=DEVICE,plots=True,
                            project=str(WORK/'eval'), name=RUN_NAME,exist_ok=True)
result_metrics={
 'mode':RUN_MODE,'scope':'public_proxy_not_center','classes':CLASS_NAMES,
 'map50':float(val_metrics.box.map50),
 'map50_95':float(val_metrics.box.map),
 'precision_mean':float(val_metrics.box.mp),
 'recall_mean':float(val_metrics.box.mr),
 'per_class_map50_95':{CLASS_NAMES[i]:float(s) for i,s in enumerate(val_metrics.box.maps)},
 'warning':'This is NOT validated on real SKKU 1398 Fitness Center photos.'
}
(DRIVE_RUN/'public_test_metrics.json').write_text(json.dumps(result_metrics,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(result_metrics,indent=2,ensure_ascii=False))

## 10. 실제 추론: 사진 → 기구명·박스·점수 JSON

`RUN_PHOTO_UPLOAD=True`로 바꾸면 본인 사진을 직접 업로드할 수 있습니다. 기본값은 공개 평가 분할의 샘플 한 장을 사용합니다. 업로드한 사진은 학습·평가 자료에 자동으로 합쳐지지 않습니다.

In [ ]:
import json
from pathlib import Path
from PIL import Image
from IPython.display import display
if RUN_PHOTO_UPLOAD:
    from google.colab import files
    uploaded=files.upload()
    if not uploaded:
        raise RuntimeError('사진이 업로드되지 않았습니다')
    photo_name=next(iter(uploaded))
    PHOTO=Path('/content')/photo_name
    PHOTO.write_bytes(uploaded[photo_name])
else:
    PHOTO=next((PREPARED/'test'/'images').iterdir())
output=MODEL_FINAL.predict(source=str(PHOTO),imgsz=IMAGE_SIZE,
                           conf=DISPLAY_CONFIDENCE,device=DEVICE,verbose=False)[0]
response=predictions_to_json(output,threshold=DISPLAY_CONFIDENCE)
print(json.dumps(response,indent=2,ensure_ascii=False))
annotated=Image.fromarray(output.plot()[...,::-1])
annotated.save(DRIVE_RUN/'single_photo_prediction.jpg')
(DRIVE_RUN/'single_photo_result.json').write_text(json.dumps(response,ensure_ascii=False,indent=2),encoding='utf-8')
display(annotated)

## 11. 모바일 파일 변환 (선택)

아래 단계는 `RUN_MOBILE_EXPORT=True`일 때만 실행합니다. 현재 Ultralytics는 `format="litert"`로 `.tflite` 형식의 모델을 내보낼 수 있습니다. 하지만 **변환 성공은 Flutter 실기기 성공과 다르며**, 앱팀의 런타임에서 전처리·출력 해석을 별도 비교해야 합니다.

In [ ]:
from pathlib import Path
import json
if RUN_MOBILE_EXPORT:
    tflite_path=Path(MODEL_FINAL.export(format='litert',imgsz=IMAGE_SIZE))
    shutil.copy2(tflite_path, DRIVE_ROOT/'04_exports_and_reports'/tflite_path.name)
    print('모바일 변환 파일:', DRIVE_ROOT/'04_exports_and_reports'/tflite_path.name)
else:
    print('모바일 변환 건너뜀. 학습 확인 후 RUN_MOBILE_EXPORT=True로 전환하세요.')

## 12. 실행 보고서 저장

최종 보고서에는 **실제 실행된 범위**가 기록됩니다. 센터 사진 추가 수집과 독립 테스트를 수행하기 전에는 **정식 현장용 모델로 납품하면 안 됩니다**.

In [ ]:
import json
from datetime import datetime
summary={
 'run_name':RUN_NAME,'created_local_time':datetime.now().isoformat(timespec='seconds'),
 'source':'FitFuel / All Gym Equipment v2 (CC BY 4.0)' if not DATASET_ZIP_NAME else str(DATASET_ZIP_NAME),
 'classes':CLASS_NAMES, 'image_size':IMAGE_SIZE, 'epochs_requested':EPOCHS,
 'run_mode':RUN_MODE, 'trained_model':str(DRIVE_RUN/'best.pt'),
 'data_audit':str(DRIVE_RUN/'dataset_audit.json'),
 'test_metrics':str(DRIVE_RUN/'public_test_metrics.json'),
 'next_step':'Real SKKU 1398 equipment photos + independent session-wise held-out evaluation; Flutter on-device validation',
 'not_claimed':'No accuracy claim for SKKU gym without site photos'
}
(DRIVE_RUN/'run_report.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('실행 결과가 Google Drive에 저장됐습니다:',DRIVE_RUN)